# Notebook 10: Citation Dynamics
We examine how citations differ **over time** between sharing status (*sharing vs. non-sharing*) and across sharing granularities (*fixation-* vs. *trial-* vs. *participant-level*).<br>
We calculate the number of citations for each article in every calendar year post-publication (ignoring "year 0", the publication year itself, and every calendar year after the last complete one, 2025, due to partial year effects). We then examine the differences in year-by-year citations across sharing categories while accounting for other article-level covariates (e.g., U.S. affiliation, venue impact score, etc.). We do so by fitting a mixed-effects model regressing *Citations in year `k` post publication* on the article covariates, including a `C(Is Sharing) * k` interaction term to test whether the citations differ between *sharing* and *non-sharing* over time (i.e. the citation **dynamics**). The same is to be done for the subset of sharing articles, testing whether a `C(Sharing Class) * k` interaction term is significant, which would indicate different citation dynamics for different shared granularities (not implemented yet).
<br><br>
**Notes:**
1. We use citation *flow* (count per year) rather than *stock* (cumulative count). A cumulative count contains all of the earlier years and can only increase, so its successive values are strongly autocorrelated, far beyond what a random intercept per article can absorb. The flow in each year is also correlated within an article, but a random intercept captures that.
2. `k` is discrete but equally spaced, so we treat it as a *continuous* variable (not a *categorical* one) in the mixed-effects model. This reduces the number of parameters, increases statistical power, and means the interaction term directly gives us the difference in slope between sharing and non-sharing articles (or between sharing granularities) over time.
3. The linear-`k` models use the raw `k` (in years), so the coefficients are per year. Because `k = 0` (the publication year) is excluded from the data, the **main effect of sharing is the group difference at `k = 0`**, outside the observed range, and is not interpreted: the sharing-by-time interaction is the test. Only the quadratic variants use standardized `k` (`k_z`, mean `0` and `SD=1`), which lets `lme4` converge on them (see below).
4. The sharing-by-time test is two-sided: unlike the sharing hypothesis in the other notebooks, we have no directional hypothesis about how the groups diverge over time.

### Model Specification
We use R's `lme4` package to fit a family of LMEs, with the following specification:
$$ f(\text{Citations}_{i,k}) \sim \text{Is Sharing} \times k + g(k) + \text{Has US Author} + \text{Is OA} + \text{Has Preprint} + log(\text{Number of Authors}) + \text{Impact Score} + C(\text{Publication Year}) + (1 | \text{Venue} / \text{Article}) $$

Here, $f()$ stands for how the citation count enters each of four models: a Gaussian on $log(1 + \text{Citations})$, a Gaussian on the raw counts, and Poisson and negative-binomial models, which model the mean count through a $log$ link instead of transforming the outcome. We use these four to check that the conclusion does not depend on the distributional assumption for skewed, zero-heavy yearly counts.<br>
We use the Gaussian on $log(1 + \text{Citations})$ as the primary model, chosen a priori. `Thelwall & Wilson (2014)` found that citation counts are well described by a discrete lognormal distribution, and recommend adding one, taking the log, and using the general linear model. Their simulations cover the regime of our data (our yearly flow fits a discrete lognormal with $\mu = 0.63$, $\sigma = 1.19$): there $log(1 + y)$ keeps the nominal 5% false-positive rate and is the most powerful trustworthy option, while negative-binomial regression falsely flags factors about 3x too often at log-SD 1 and about half the time at log-SD 2, worsening with sample size. The discrete lognormal itself has no mixed-model implementation, so $log(1 + y)$ is the stand-in. One caveat is that their simulations were for single-level models, not for repeated measures.<br>
The other three families are used as robustness checks, to be mentioned in the discussion section of the manuscript. They do not estimate the same quantity: the Gaussian on $log(1 + y)$ estimates a proportional change in the geometric mean of $1 + y$, while Poisson and negative binomial estimate a rate ratio of the arithmetic mean. We therefore compare the families on the sign and significance of the sharing-by-time coefficient, not on its size.
<br><br>
The $g(k)$ term is a non-linear function of $k$ (e.g., $log(k)$ or $k^2$) to account for non-linear effects of time on citations. We compare the linear model against models that allow non-linear variations, and keep the linear term unless another lowers the AIC by more than 2 (the usual margin for "equivalent" fits).
<br><br>
We use $(1 | \text{Venue} / \text{Article})$ to account for the nested structure of the data: we have many observations of the same article (over different years); and articles are in turn nested in venues. The random intercepts for articles account for the correlation between an article's repeated yearly counts, while the random intercepts for venues account for the correlation between articles in the same venue.<br>
**Limitations of this structure:**
- A random intercept implies the same correlation between any two years of an article, and the model has no article-specific time trends (random slopes for `k`) or serial correlation. Since sharing varies between articles and `k` within them, omitting random slopes can make the SE of the sharing-by-time interaction too small. This is a limitation to state, or to check with `(1 + k_z | article)`.
- Publication year enters as a categorical fixed effect, and calendar year is publication year plus `k`, so any linear calendar-year trend is absorbed into the `k` slope. Later cohorts also contribute fewer years, so the high-`k` tail is dominated by older articles with few sharers (in the superseded analysis, N = 234 at year 1 fell to 84 by year 7, with 19 sharers left). This is why the predictions are limited to years with enough articles (see below).


In [1]:
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from helpers.config import *
from helpers import dataset
from helpers.plotting import save_figure
from helpers.r_models import fit_glmm

pio.renderers.default = "browser"

## Setup

In [2]:
# the four fits: (R family, outcome). The Gaussian on log(1 + citations) is the primary (see the specification above);
# Poisson and negative binomial are robustness fits on the raw counts, and the raw Gaussian is kept for reference
DYNAMICS_FAMILIES = {
    "gaussian (log1p)": ("gaussian", "log_citations"),
    "gaussian (raw)": ("gaussian", "citations"),
    "poisson": ("poisson", "citations"),
    "nb": ("nb", "citations"),
}
PRIMARY_FAMILY = "gaussian (log1p)"


# set up a generic formula, which can be edited using the `formula_for()` helper
DYNAMICS_FORMULA = (
    "citations ~ is_sharing_data * k + has_us_author + is_open_access + has_preprint"
    " + log_number_of_authors + venue_impact + factor(publication_year) + (1 | venue_name/article)"
)

def formula_for(outcome: str, formula: str = DYNAMICS_FORMULA) -> str:
    return formula.replace("citations ~", f"{outcome} ~", 1)

In [3]:
def citations_per_year(
        articles: pd.DataFrame, grouping_col: str, cumulative: bool = False
) -> pd.DataFrame:
    """Citations per year since publication, indexed by (article, group).

    The reshape from absolute calendar years to years-since-publication is done by
    `dataset.citations_since_publication()`. This wrapper only applies the two analysis choices
    belonging to this notebook, which that helper deliberately leaves to its callers: drop the
    publication year itself (`year_0`, partial for every article), and drop any calendar year that
    has not fully elapsed for the whole sample.

    The incomplete-year cutoff comes from `dataset.citations_through_shared_year()` rather than a
    hard-coded year, so it tracks the data instead of going stale at the next re-fetch.

    :param articles: frame carrying the `Citations20XX` columns, `PublicationYear`, `LastUpdate`
        and `grouping_col`.
    :param grouping_col: column to append as the second index level, e.g. `is_sharing_data`.
    :param cumulative: True for cumulative citations (stock), False for per-year counts (flow).
    :return: frame indexed by (article, `grouping_col`), one column per year offset.
    """
    _, shared_year = dataset.citations_through_shared_year(articles)
    incomplete = [
        col for col in articles.columns
        if col.startswith("Citations") and col.removeprefix("Citations").isdigit()
        and int(col.removeprefix("Citations")) > shared_year
    ]
    per_year = dataset.citations_since_publication(articles.drop(columns=incomplete))
    per_year = per_year.drop(columns="year_0")
    per_year.columns = [int(col.removeprefix("year_")) for col in per_year.columns]
    if cumulative:
        per_year = per_year.cumsum(axis=1)
    return per_year.set_index(articles[grouping_col], append=True)

### Load Data

In [4]:
combined, FEATURES_DF = dataset.load_or_build()

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\data_store (234 rows)


## Sharing vs. Non-Sharing

In [5]:
GROUPING_VARIABLE = "is_sharing_data"
INTERACTION = "is_sharing_data:k"

In [6]:
per_year = (
    citations_per_year(combined, grouping_col=GROUPING_VARIABLE, cumulative=False)
    .droplevel(GROUPING_VARIABLE)
    .stack()
    .rename("citations")
    .dropna()                               # years an article has not reached yet
)
per_year.index.names = ["article", "k"]     # k is the year offset since publication

# long format: one row per article-year, with the article's covariates repeated across its rows
# (publication year is OpenAlex's `PublicationYear`, not Godwin et al.'s `YEAR_PUBLISHED`)
article_covariates = dataset.to_smf_columns(FEATURES_DF).assign(publication_year=combined["PublicationYear"].astype(int))
DYNAMICS_DF = (
    per_year.reset_index()
    .merge(article_covariates, left_on="article", right_index=True, validate="many_to_one")
)
DYNAMICS_DF

,article,k,citations,is_sharing_data,sharing_class,has_us_author,is_open_access,has_preprint,log_weeks_since_pub,log_number_of_authors,venue_impact,venue_name,publication_year
0,0,1,2.0,1,FIXATION,0,1,1,5.653992,1.098612,0.429831,Journal of Vision,2020
1,0,2,3.0,1,FIXATION,0,1,1,5.653992,1.098612,0.429831,Journal of Vision,2020
2,0,3,0.0,1,FIXATION,0,1,1,5.653992,1.098612,0.429831,Journal of Vision,2020
3,0,4,1.0,1,FIXATION,0,1,1,5.653992,1.098612,0.429831,Journal of Vision,2020
4,0,5,1.0,1,FIXATION,0,1,1,5.653992,1.098612,0.429831,Journal of Vision,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1342,250,8,2.0,0,NONE,1,1,0,6.134018,0.693147,0.429831,Journal of Vision,2017
1343,251,1,2.0,0,NONE,1,1,0,5.463226,1.098612,3.469027,Cognitive Research Principles and Implications,2021
1344,251,2,3.0,0,NONE,1,1,0,5.463226,1.098612,3.469027,Cognitive Research Principles and Implications,2021
1345,251,3,2.0,0,NONE,1,1,0,5.463226,1.098612,3.469027,Cognitive Research Principles and Implications,2021


### Multi-Variable Models
#### Prepare Data

In [ ]:
MIN_ARTICLES_PER_K = 20
K_MEAN, K_SD = DYNAMICS_DF["k"].mean(), DYNAMICS_DF["k"].std()

MODEL_DATA = DYNAMICS_DF.assign(
    log_k=np.log(DYNAMICS_DF["k"]),
    k_z=(DYNAMICS_DF["k"] - K_MEAN) / K_SD,
    log_citations=np.log1p(DYNAMICS_DF["citations"]),
)

In [ ]:
# typical-article predictions (covariates at their mean / mode, random effects at zero), for each year with at least
# `MIN_ARTICLES_PER_K` articles: later years have few articles, and fewer sharers, so the curves would only extrapolate
articles_per_k = DYNAMICS_DF.groupby("k")["article"].nunique()
PREDICTION_GRID = pd.DataFrame(
    [{"is_sharing_data": sharing, "k": k} for sharing in (0, 1) for k in articles_per_k.index[articles_per_k >= MIN_ARTICLES_PER_K]]
)
for covariate in ["has_us_author", "is_open_access", "has_preprint", "log_number_of_authors", "venue_impact"]:
    PREDICTION_GRID[covariate] = DYNAMICS_DF[covariate].mean()
PREDICTION_GRID["publication_year"] = DYNAMICS_DF["publication_year"].mode()[0]
PREDICTION_GRID["log_k"] = np.log(PREDICTION_GRID["k"])
PREDICTION_GRID["k_z"] = (PREDICTION_GRID["k"] - K_MEAN) / K_SD

#### Fit Models

In [7]:
dynamics_fits = {
    name: fit_glmm(MODEL_DATA, formula_for(outcome), family, newdata=PREDICTION_GRID)
    for name, (family, outcome) in DYNAMICS_FAMILIES.items()
}

#### Model Comparison

In [26]:
def compare_families(fits: dict) -> pd.DataFrame:
    rows = {}
    for name, result in fits.items():
        effect = result.fixed_effects.loc[INTERACTION]
        is_count_family = result.family != "gaussian"
        rows[name] = {
            "converged": result.fit["converged"], "singular": result.fit["singular"],
            # a Gaussian density is not comparable to a count pmf, nor to a Gaussian on another outcome: AIC only for counts
            "log-lik": result.fit["log_lik"], "AIC": result.fit["aic"] if is_count_family else np.nan,
            "overdispersion": result.fit["overdispersion"], "NB theta": result.fit["theta"],
            "b": effect["estimate"], "SE": effect["se"], "p": effect["p"],
            "% change per year": 100 * (effect["rate_ratio"] - 1) if "rate_ratio" in effect.index else np.nan,
        }
    return pd.DataFrame(rows).T

In [8]:
family_comparison = compare_families(dynamics_fits)

# only Poisson and negative binomial compete on AIC: it picks the count family for the robustness fit, not the primary
count_aic = family_comparison["AIC"].dropna()
best_count_family = count_aic.idxmin()
print(f"Poisson overdispersion ratio: {family_comparison.loc['poisson', 'overdispersion']:.2f}")
print(f"Best count family by AIC: {best_count_family} (AIC difference: {count_aic.max() - count_aic.min():.1f})")
print(f"Primary family (chosen a priori, not by AIC): {PRIMARY_FAMILY}")

family_comparison

Poisson overdispersion ratio: 1.05
Best count family by AIC: nb (AIC difference: 62.4)
Primary family (chosen a priori, not by AIC): gaussian (log1p)


,converged,singular,log-lik,AIC,overdispersion,NB theta,b,SE,p,% change per year
gaussian (log1p),True,False,-1188.071365,NaN,NaN,NaN,-0.041589,0.015978,0.009361,NaN
gaussian (raw),True,False,-3292.370383,NaN,NaN,NaN,-0.210509,0.075824,0.005587,NaN
poisson,True,False,-2801.373334,5636.746668,1.053903,NaN,-0.057787,0.016996,0.000674,-5.61492
nb,True,False,-2769.151774,5574.303548,0.80723,11.530452,-0.062915,0.020794,0.002481,-6.097718


### Model Results
#### Primary Family Results
We report the results from the Gaussian `log(1 + citations)` fit with linear `k` (in raw years, not standardized). The outcome is on the log scale, so `exp(b) - 1` for the sharing-by-time coefficient is the proportional difference per year in the yearly citations (of `1 + citations`) between sharing and non-sharing articles. The time term used for the figure is chosen below.


In [34]:
primary_fit = dynamics_fits[PRIMARY_FAMILY]
print(f"The {PRIMARY_FAMILY} model fit converged: {primary_fit.fit['converged']}, singular: {primary_fit.fit['singular']}")

interaction_b = primary_fit.fixed_effects.loc[INTERACTION, "estimate"]
interaction_p = primary_fit.fixed_effects.loc[INTERACTION, "p"]
is_interaction_significant = interaction_p < 0.05
print(f"The sharing-by-time interaction term is {'significant' if is_interaction_significant else 'not significant'} (interaction_b = {interaction_b:.3f}, p = {interaction_p:.3f})")

print("Fixed Effects:")
display(primary_fit.fixed_effects)

print("Random Effects:")
display(primary_fit.variance_components)

The gaussian (log1p) model fit converged: True, singular: False
The sharing-by-time interaction term is significant (interaction_b = -0.042, p = 0.009)
Fixed Effects:


,estimate,se,statistic,p
term,,,,
(Intercept),0.843615,0.246814,3.418022,0.000842
is_sharing_data,0.142899,0.106773,1.338344,0.181598
k,0.003133,0.008378,0.373975,0.708493
has_us_author,0.109238,0.079567,1.372913,0.171113
is_open_access,0.202300,0.122057,1.657420,0.099724
has_preprint,0.125237,0.102089,1.226742,0.221142
log_number_of_authors,0.032715,0.094665,0.345588,0.729966
venue_impact,0.124044,0.044604,2.780983,0.012277
factor(publication_year)2017,-0.249393,0.206699,-1.206552,0.228958


Random Effects:


,group,term,variance,sd
0,article:venue_name,(Intercept),0.283245,0.532208
1,venue_name,(Intercept),0.024567,0.156739
2,Residual,NaN,0.238480,0.488345


#### Robustness Check: Sensitivity to the Offset in `log(y + c)`
We report the results from the Gaussian `log(1 + citations)` fit, but the offset of 1 is arbitrary. We refit the model with different offsets and with the zeros dropped (no offset at all) to check whether the sharing-by-time coefficient is sensitive to this choice.<br>
The coefficients are on different scales for different offsets, so we compare them by `|t|` rather than by the estimate itself.<br>
Dropping the zeros also changes the sample and the question: it conditions on the article being cited that year, so it checks the direction and strength of the effect among cited article-years, not the same quantity.


In [10]:
# sensitivity of the sharing-by-time coefficient to the offset in log(y + c): the coefficients are on different scales, so compare |t|
offset_fits = {
    f"log(y + {offset})": fit_glmm(
        MODEL_DATA.assign(outcome=np.log(DYNAMICS_DF["citations"] + offset)),
        formula_for("outcome"),
        "gaussian",
    )
    for offset in (1.0, 0.5, 0.25, 0.1)
}
positive = MODEL_DATA[MODEL_DATA["citations"] > 0]
offset_fits["log(y), zeros dropped"] = fit_glmm(
    positive.assign(outcome=np.log(positive["citations"])), formula_for("outcome"), "gaussian",
)

offset_sensitivity = pd.DataFrame({
    name: {**fit.fixed_effects.loc[INTERACTION, ["estimate", "se", "p"]].astype(float).to_dict(), "N": fit.fit["n_obs"]}
    for name, fit in offset_fits.items()
}).T
offset_sensitivity["|t|"] = (offset_sensitivity["estimate"] / offset_sensitivity["se"]).abs()
offset_sensitivity

,estimate,se,p,N,|t|
log(y + 1.0),-0.041589,0.015978,0.009361,1347.0,2.602891
log(y + 0.5),-0.051438,0.020759,0.013358,1347.0,2.477848
log(y + 0.25),-0.060899,0.026071,0.019665,1347.0,2.335873
log(y + 0.1),-0.072766,0.033804,0.031551,1347.0,2.152605
"log(y), zeros dropped",-0.041789,0.019932,0.036295,1117.0,2.096552


#### Robustness Check: Sensitivity to the Count Family
We check the sensitivity of the results to the choice of count family by reporting the sharing-by-time coefficient from the better-fitting count family (Poisson or negative binomial, compared by AIC, which is comparable between these two) as a rate ratio. This is a different quantity from the one in our a-priori chosen Gaussian `log(1 + citations)` model (a ratio of arithmetic mean counts, not a proportional change in the geometric mean of `1 + citations`), so we compare the two on direction and significance, not on size.


In [11]:
# robustness: the count family with the lower AIC, as rate ratios
count_fit = dynamics_fits[best_count_family]
print(f"Robustness fit ({best_count_family}), rate ratios:")
count_fit.fixed_effects

Robustness fit (nb), rate ratios:


,estimate,se,statistic,p,rate_ratio,rr_ci_low,rr_ci_high
term,,,,,,,
(Intercept),0.285934,0.364271,0.784947,0.432485,1.331004,0.651795,2.717989
is_sharing_data,0.224744,0.151123,1.487163,0.136972,1.252002,0.931042,1.683607
k,0.011539,0.011639,0.991457,0.321462,1.011606,0.988791,1.034948
has_us_author,0.156606,0.117154,1.336755,0.181303,1.169535,0.929590,1.471413
is_open_access,0.314614,0.180056,1.747308,0.080584,1.369731,0.962434,1.949394
has_preprint,0.132826,0.148452,0.894742,0.370925,1.142051,0.853736,1.527734
log_number_of_authors,0.085051,0.139020,0.611789,0.540678,1.088772,0.829093,1.429785
venue_impact,0.174286,0.070553,2.470286,0.013501,1.190396,1.036661,1.366929
factor(publication_year)2017,-0.339752,0.300818,-1.129428,0.258717,0.711947,0.394813,1.283820


### Non-Linear Time
We check if non-linear time (e.g. $log(k)$ or $k^2$) fits better than linear $k$. In the variants `k + log(k)` and `k + k^2` the non-linear term is shared by both groups, so the sharing-by-time interaction stays linear in time (no $\text{Is Sharing} \times log(k)$ or $\text{Is Sharing} \times k^2$) and one coefficient summarizes it. The `log(k)` and `k^2` variants instead interact sharing with the transformed term.<br>
The two `(main only)` variants combine the linear sharing-by-time interaction with a non-linear main term but **without the main effect of `k` itself**, so their interaction coefficient is the sharing group's whole linear slope, not a difference in slopes. Only their AIC is comparable to the other models, not the meaning of the coefficient.
<br><br>
We use standardized $k$ (`k_z`) in the quadratic variants because $k$ and $k^2$ are nearly collinear, which stops `lme4` from converging. For `k + k^2` this is the same model as with raw $k$, only reparametrized: the slope is per SD of $k$, and the sharing main effect refers to the mean $k$.
<br><br>
Since we are using the same family, outcome, and data, the AIC is comparable across these models, provided they are fitted by maximum likelihood (they are: REML likelihoods cannot be compared across different fixed effects).


In [35]:
TIME_TERMS = {
    "log(k)": "is_sharing_data * log_k",
    "k + log(k)": "is_sharing_data * k + log_k",
    "log(k) (main only)": "is_sharing_data + log_k + is_sharing_data:k",
    "k^2": "is_sharing_data * I(k_z^2)",
    "k + k^2": "is_sharing_data * k_z + I(k_z^2)",
    "k^2 (main only)": "is_sharing_data + I(k_z^2) + is_sharing_data:k_z",
}
primary_family, primary_outcome = DYNAMICS_FAMILIES[PRIMARY_FAMILY]

time_fits = {"k": primary_fit} | {
    name: fit_glmm(
        MODEL_DATA,
        formula_for(primary_outcome, DYNAMICS_FORMULA.replace("is_sharing_data * k", term)),
        primary_family,
        newdata=PREDICTION_GRID,
    )
    for name, term in TIME_TERMS.items()
}

Lowest AIC: k + log(k) (-16.2 vs. k): using k + log(k)


,converged,AIC,AIC vs. k
k,True,2412.142730,0.000000
log(k),True,2412.691807,0.549078
k + log(k),True,2395.938699,-16.204031
log(k) (main only),True,2408.701598,-3.441132
k^2,True,2412.052297,-0.090432
k + k^2,True,2405.882701,-6.260029
k^2 (main only),True,2405.640072,-6.502657


,,estimate,se,p
time term,term,,,
k,is_sharing_data:k,-0.041589,0.015978,0.009361
log(k),is_sharing_data:log_k,-0.130208,0.047695,0.006429
k + log(k),is_sharing_data:k,-0.047418,0.015908,0.002935
log(k) (main only),is_sharing_data:k,-0.053944,0.015919,0.000726
k^2,is_sharing_data:I(k_z^2),0.000777,0.027781,0.977693
k + k^2,is_sharing_data:k_z,-0.092961,0.032502,0.004309
k^2 (main only),is_sharing_data:k_z,-0.070858,0.027933,0.011319


In [43]:
# model comparison
time_comparison = pd.DataFrame.from_dict(
    {name: {"converged": fit.fit["converged"], "AIC": fit.fit["aic"]} for name, fit in time_fits.items()},
    orient="index",
)
time_comparison["AIC vs. k"] = time_comparison["AIC"] - time_comparison.loc["k", "AIC"]

# keep the simplest term (k) unless another improves the AIC by more than 2, the usual margin for "equivalent" fits
lowest_aic = time_comparison["AIC"].idxmin()
best_time_term = lowest_aic if time_comparison.loc[lowest_aic, "AIC vs. k"] < -2 else "k"
print(f"Lowest AIC: {lowest_aic} ({time_comparison.loc[lowest_aic, 'AIC vs. k']:+.1f} vs. k): using {best_time_term}")
time_comparison

Lowest AIC: k + log(k) (-16.2 vs. k): using k + log(k)


,converged,AIC,AIC vs. k
k,True,2412.142730,0.000000
log(k),True,2412.691807,0.549078
k + log(k),True,2395.938699,-16.204031
log(k) (main only),True,2408.701598,-3.441132
k^2,True,2412.052297,-0.090432
k + k^2,True,2405.882701,-6.260029
k^2 (main only),True,2405.640072,-6.502657


In [36]:
# the sharing-by-time terms of each model
nonlinear_k_interactions = pd.concat(
    {name: fit.fixed_effects.filter(like="is_sharing_data:", axis=0)[["estimate", "se", "p"]] for name, fit in time_fits.items()},
    names=["time term", "term"],
)

print(f"The sharing-by-time interaction term is significant in {sum(nonlinear_k_interactions['p'] < 0.05)} / {len(nonlinear_k_interactions)} models:")
nonlinear_k_interactions

The sharing-by-time interaction term is significant in 6 / 7 models:


,,estimate,se,p
time term,term,,,
k,is_sharing_data:k,-0.041589,0.015978,0.009361
log(k),is_sharing_data:log_k,-0.130208,0.047695,0.006429
k + log(k),is_sharing_data:k,-0.047418,0.015908,0.002935
log(k) (main only),is_sharing_data:k,-0.053944,0.015919,0.000726
k^2,is_sharing_data:I(k_z^2),0.000777,0.027781,0.977693
k + k^2,is_sharing_data:k_z,-0.092961,0.032502,0.004309
k^2 (main only),is_sharing_data:k_z,-0.070858,0.027933,0.011319


### Visualize Model Predictions
We use the model with the best time term (linear or non-linear) to predict the expected citation counts per year for sharing vs. non-sharing articles, and plot the predicted citation dynamics.<br>
The predictions are for a typical article (covariates at their mean or mode, random effects at zero), for each year with at least `MIN_ARTICLES_PER_K` articles. For the `log(1 + y)` model they are back-transformed to the geometric-mean scale (geometric mean of `1 + y`, minus 1), so they sit below the arithmetic mean of the counts.


In [42]:
predictions = time_fits[best_time_term].predictions.copy()
if primary_outcome == "log_citations":      # back-transform from log(1 + y): the geometric-mean scale, not the arithmetic mean
    predictions[["fit", "ci_low", "ci_high"]] = np.expm1(predictions[["fit", "ci_low", "ci_high"]])

dynamics_fig = go.Figure()
for is_sharing, label in ((1, "SHARING"), (0, "NOT SHARING")):
    subset = predictions[predictions["is_sharing_data"] == is_sharing]
    color = CLASS_COLORS[label]
    dynamics_fig.add_trace(go.Scatter(                  # confidence band
        x=pd.concat([subset["k"], subset["k"][::-1]]),
        y=pd.concat([subset["ci_high"], subset["ci_low"][::-1]]),
        mode="lines", fill="toself", fillcolor=color, opacity=0.2, line=dict(width=0),
        hoverinfo="skip", showlegend=False, legendgroup=label,
    ))
    dynamics_fig.add_trace(go.Scatter(
        x=subset["k"], y=subset["fit"], name=label.title(), legendgroup=label,
        mode="lines+markers", line=dict(color=color), marker=dict(color=color),
    ))

# update layout and styling
dynamics_fig.update_xaxes(
    title=dict(text="Years Since Publication", font=AXIS_TITLE_FONT, standoff=10),
    tickfont=AXIS_TICK_FONT, tickmode="linear", dtick=1,
)
dynamics_fig.update_yaxes(
    title=dict(text="Predicted Citations per Year (geometric mean)", font=AXIS_TITLE_FONT, standoff=15),
    tickfont=AXIS_TICK_FONT, rangemode="tozero",
)
dynamics_fig.update_layout(
    width=800, height=400,
    title=dict(text="Citation Dynamics: Sharing vs. Non-Sharing", font=TITLE_FONT, x=0.5, xanchor="center"),
    legend=dict(font=LEGEND_FONT),
    margin=dict(t=60, b=0, l=0, r=0),
    template="plotly_white",
)

if False:
    dynamics_fig.show()

In [14]:
# flip to True to export this figure into `output/`
if False:
    save_figure(dynamics_fig, "citation_dynamics_model.png", width=800, height=400)

In [15]:
1/0     # todo: drop this when working on sharing granularities

ZeroDivisionError: division by zero

---

## Sharing Granularity

In [ ]:
GROUPING_VARIABLE = "sharing_class"

DYNAMICS_GRANULARITY_DF = DYNAMICS_DF.loc[DYNAMICS_DF[GROUPING_VARIABLE] != "NONE"]
DYNAMICS_GRANULARITY_DF[GROUPING_VARIABLE] = DYNAMICS_GRANULARITY_DF[GROUPING_VARIABLE].cat.remove_unused_categories()

# verify we are left only with the three sharing classes
DYNAMICS_GRANULARITY_DF[GROUPING_VARIABLE].unique()

In [ ]:
# TODO

In [ ]:
# TODO: drop this to see the (deprecated) residuals analysis
1/0

---

## Citation Dynamics :: Univariate Comparisons (SUPERSEDED)
**This section is superseded and does not appear in the revised manuscript.** It is the analysis
reported in the submitted version, kept so the published v1 results stay reproducible. Everything
from this heading to the end of the notebook belongs to it, including the per-group comparisons at
the bottom.
<br><br>
We explor how citations differ **over time** between sharing and non-sharing articles; and whether different types of data sharing are associated with different citation dynamics.<br>
To do so, we calculate the number of citations in each calendar year post-publication for each article - ignoring "year 0" and the current year (2026) due to partial year effects. We then examine the differences in year-by-year citations across sharing categories (*sharing vs. non-sharing* and *sharing granularities*). We use a set of Mann-Whitney U-tests to compare the distributions of citations per year between groups, and correct for multiple comparisons using the Benjamini-Hochberg FDR procedure.

### (1) Sharing vs. Non-Sharing Articles
We compare the cumulative citation counts per year between articles that share data vs. those that do not.

In [ ]:
GROUPING_COL = "is_sharing_data"

# cumulative shows the total citations up to (and including) each year (stock)
# non-cumulative shows the "velocity" of citations per year (flow)
cit_dynamics_sharing = citations_per_year(
    combined, grouping_col=GROUPING_COL,
    cumulative=True
)
# drop years with fewer than 20 articles combined
cit_dynamics_sharing = cit_dynamics_sharing.loc[:, cit_dynamics_sharing.count(axis=0) >= 20]

year_stats = dict()
for year in cit_dynamics_sharing.columns:
    non_sharing_vals = cit_dynamics_sharing.xs(False, level=GROUPING_COL)[year].dropna()
    non_sharing_is_normal = stats.shapiro(non_sharing_vals).pvalue > 0.05
    sharing_vals = cit_dynamics_sharing.xs(True, level=GROUPING_COL)[year].dropna()
    sharing_is_normal = stats.shapiro(sharing_vals).pvalue > 0.05
    n1, n2 = len(non_sharing_vals), len(sharing_vals)
    yearly_ttest = pg.ttest(non_sharing_vals, sharing_vals, paired=False, alternative="less", correction=True)
    yearly_mwtest = pg.mwu(non_sharing_vals, sharing_vals, alternative="less")
    U_nonshare = yearly_mwtest.loc["MWU", "U_val"]
    U_share = n1 * n2 - U_nonshare
    rank_biserial_corr = (U_share - U_nonshare) / (n1 * n2)
    cles = U_share / (n1 * n2)
    year_stats[year] = {
        "non_sharing_count": n1,
        "non_sharing_is_normal": non_sharing_is_normal,
        "non_sharing_median": non_sharing_vals.median(),
        "non_sharing_mean": non_sharing_vals.mean(),
        "non_sharing_sem": non_sharing_vals.sem(),

        "sharing_count": n2,
        "sharing_is_normal": sharing_is_normal,
        "sharing_median": sharing_vals.median(),
        "sharing_mean": sharing_vals.mean(),
        "sharing_sem": sharing_vals.sem(),

        "t_stat": yearly_ttest.loc["T_test", "T"],
        "t_dof": yearly_ttest.loc["T_test", "dof"],
        "t_p_raw": yearly_ttest.loc["T_test", "p_val"],
        "t_cohen_d": yearly_ttest.loc["T_test", "cohen_d"],

        "U_nonshare": U_nonshare,
        "U_share": U_share,
        "mw_p_raw": yearly_mwtest.loc["MWU", "p_val"],
        "r_rb": abs(yearly_mwtest.loc["MWU", "RBC"]),
        "cles": yearly_mwtest.loc["MWU", "CLES"],
    }

year_stats = pd.DataFrame.from_dict(year_stats, orient='index').reset_index().rename(columns={"index": "year"})
t_reject, t_p_corrected, _, _ = multipletests(year_stats['t_p_raw'], alpha=0.05, method='fdr_bh')
mw_reject, mw_p_corrected, _, _ = multipletests(year_stats['mw_p_raw'], alpha=0.05, method='fdr_bh')
year_stats['t_p_corrected'] = t_p_corrected
year_stats['t_is_significant'] = t_reject
year_stats['mw_p_corrected'] = mw_p_corrected
year_stats['mw_is_significant'] = mw_reject
column_order = [
    "year",
    "non_sharing_is_normal", "sharing_is_normal",
    "t_is_significant", "t_p_corrected",
    "mw_is_significant", "mw_p_corrected",
    "t_stat", "t_dof", "t_p_raw", "t_cohen_d",
    "U_share", "U_nonshare", "r_rb", "cles",
    "non_sharing_count", "non_sharing_mean", "non_sharing_sem", "non_sharing_median",
    "sharing_count", "sharing_mean", "sharing_sem", "sharing_median",
]
year_stats = year_stats[column_order]

year_stats

In [ ]:
LINE_ANNOTATION_X = 3

cit_dynamics_sharing_fig = go.Figure()
for is_share in cit_dynamics_sharing.index.get_level_values(GROUPING_COL).unique().sort_values():
    subset = cit_dynamics_sharing.xs(is_share, level=GROUPING_COL)
    count_per_year = subset.count(axis=0)
    mean_per_year = subset.mean(axis=0)
    sem_per_year = subset.sem(axis=0)
    mean_citations_per_year = subset.mean(axis=0)
    label = "Sharing" if is_share else "Not Sharing"
    color = CLASS_COLORS[label.upper()]
    cit_dynamics_sharing_fig.add_trace(go.Scatter(
        name=label, legendgroup=label,
        x=mean_per_year.index,
        y=mean_per_year.values,
        error_y=dict(type='data', array=sem_per_year.values, width=5, visible=True,),
        marker=dict(
            color=color,
            # size=np.sqrt(count_per_year.values),
        ),
        mode='lines+markers',
    ))
    # annotate the lines
    if is_share:
        ann_y = mean_per_year[LINE_ANNOTATION_X] + 1.5 * sem_per_year[LINE_ANNOTATION_X]
        xanchor = "right"
        yanchor = "bottom"
    else:
        ann_y = mean_per_year[LINE_ANNOTATION_X] - 1.5 * sem_per_year[LINE_ANNOTATION_X]
        xanchor = "left"
        yanchor = "top"
    cit_dynamics_sharing_fig.add_annotation(
        x=LINE_ANNOTATION_X, xanchor=xanchor, xref="x",
        y=ann_y, yanchor=yanchor, yref="y",
        text=label, font={**AXIS_TITLE_FONT, "color": color},
        showarrow=False,
    )

# add significance asterisks
SIGNIFICANCE_ANNOTATION_Y = 40
for _, row in year_stats.iterrows():
    if row["mw_p_corrected"] >= 0.1:
        continue
    fontsize = 22
    yanchor = "middle"
    if row["mw_p_corrected"] < 0.001:
        ann_text = "<b>***</b>"
    elif row["mw_p_corrected"] < 0.01:
        ann_text = "<b>**</b>"
    elif row["mw_p_corrected"] < 0.05:
        ann_text = "<b>*</b>"
    else:
        ann_text = "\u2020"     # dagger for p < 0.
        fontsize = 16
        yanchor = "bottom"
    cit_dynamics_sharing_fig.add_annotation(
        x=row["year"], xanchor="center", xref="x",
        y=SIGNIFICANCE_ANNOTATION_Y, yanchor=yanchor, yref="y",
        text=ann_text, font={**TITLE_FONT, "size": fontsize},
        showarrow=False,
    )

cit_dynamics_sharing_fig.update_layout(
    width=800, height=400,
    title=dict(
        text="Citation Dynamics: Sharing vs. Non-Sharing Articles",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top"
    ),
    xaxis=dict(
        title=dict(text="Years Since Publication", font=AXIS_TITLE_FONT, standoff=10),
        tickfont=AXIS_TICK_FONT,
        zeroline=False,
    ),
    yaxis=dict(
        title=dict(text="Mean Cumulative Citation per Year", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
        showgrid=True, gridcolor='lightgrey', gridwidth=1.5,
        zeroline=False,
    ),
    legend=dict(
        title=dict(text="Data Sharing", font=AXIS_TITLE_FONT),
        font=LEGEND_FONT,
        x=1.0, xanchor="right", y=0.4, yanchor="top",
        bgcolor='rgba(255,255,255,0.5)',
        bordercolor='black', borderwidth=1,
        visible=False,
    ),
    margin=dict(t=75, b=50, l=50, r=25, pad=0),
    template="plotly_white"
)
cit_dynamics_sharing_fig.show()

In [ ]:
# flip to True to export this figure into `output/`
if False:
    save_figure(cit_dynamics_sharing_fig, "citation_dynamics_sharing_vs_nonsharing.png", width=800, height=400)

##### Source of the Difference
So we found a difference in yearly cumulative citations between sharing and non-sharing articles.<br>
But where does this difference come from? Is it because the sharing articles get cited more frequently in each year (i.e., higher "velocity" of citations)? Or is it because they got an initian citation boost which then accumulates over time (i.e., higher "stock" of citations)?<br>
To answer this question, we repeat the above analysis but now looking at **non-cumulative** citation counts per year (i.e., the "velocity" of citations).

In [ ]:
GROUPING_COL = "is_sharing_data"

# cumulative shows the total citations up to (and including) each year (stock)
# non-cumulative shows the "velocity" of citations per year (flow)
cit_dynamics_sharing = citations_per_year(
    combined, grouping_col=GROUPING_COL,
    cumulative=False
)

year_stats = dict()
for year in cit_dynamics_sharing.columns:
    non_sharing_vals = cit_dynamics_sharing.xs(False, level=GROUPING_COL)[year].dropna()
    sharing_vals = cit_dynamics_sharing.xs(True, level=GROUPING_COL)[year].dropna()
    n1, n2 = len(non_sharing_vals), len(sharing_vals)
    year_pair_test = stats.mannwhitneyu(non_sharing_vals, sharing_vals, alternative="less")
    U_nonshare = year_pair_test.statistic
    U_share = n1 * n2 - U_nonshare
    rank_biserial_corr = (U_share - U_nonshare) / (n1 * n2)
    cles = U_share / (n1 * n2)
    year_stats[year] = {
        "non_sharing_count": n1,
        "non_sharing_mean": non_sharing_vals.mean(),
        "non_sharing_sem": non_sharing_vals.sem(),
        "sharing_count": n2,
        "sharing_mean": sharing_vals.mean(),
        "sharing_sem": sharing_vals.sem(),
        "U_nonshare": U_nonshare,
        "p_raw": year_pair_test.pvalue,
        "r_rb": rank_biserial_corr,
        "cles": cles,
    }

year_stats = pd.DataFrame.from_dict(year_stats, orient='index').reset_index().rename(columns={"index": "year"})
reject, p_corrected, _, _ = multipletests(year_stats['p_raw'], alpha=0.05, method='fdr_bh')
year_stats['p_corrected'] = p_corrected
year_stats['is_significant'] = reject
column_order = [
    "year",
    "is_significant", "p_corrected", "U_nonshare", "r_rb", "cles",
    "non_sharing_count", "non_sharing_mean", "non_sharing_sem",
    "sharing_count", "sharing_mean", "sharing_sem",
    "p_raw"
]
year_stats = year_stats[column_order]

year_stats

### (2) Different Types of Data Sharing

In [ ]:
GROUPING_COL = "data_sharing_class"

cit_dynamics_sharing_class = citations_per_year(
    combined.loc[combined["is_sharing_data"]], grouping_col=GROUPING_COL
)

year_stats = dict()
for year in cit_dynamics_sharing_class.columns:
    year_stats[year] = dict()
    class_values = []
    for share_class in cit_dynamics_sharing_class.index.get_level_values(GROUPING_COL).unique():
        subset = (cit_dynamics_sharing_class.xs(share_class, level=GROUPING_COL)[year]).dropna()
        if subset.empty:
            continue
        year_stats[year][f"{share_class}_count"] = subset.count()
        year_stats[year][f"{share_class}_mean"] = subset.mean()
        year_stats[year][f"{share_class}_sem"] = subset.sem()
        class_values.append(subset)
    sharegroup_test = stats.kruskal(*class_values)
    year_stats[year]["W"] = sharegroup_test.statistic
    year_stats[year]["p_raw"] = sharegroup_test.pvalue

year_stats = pd.DataFrame.from_dict(year_stats, orient='index')
reject, p_corrected, _, _ = multipletests(year_stats['p_raw'], alpha=0.05, method='fdr_bh')
year_stats['p_corrected'] = p_corrected
year_stats['is_significant'] = reject
column_order = [
    "is_significant", "p_corrected", "W", "p_raw",
    *[f"{share_class}_{stat}" for share_class in ["FIXATION", "TRIAL", "PARTICIPANT"] for stat in ["count", "mean", "sem"]]
]
year_stats = year_stats[column_order]
year_stats

In [ ]:
cit_dynamics_sharing_class_fig = go.Figure()
for share_class in cit_dynamics_sharing_class.index.get_level_values(GROUPING_COL).unique():
    subset = cit_dynamics_sharing_class.xs(share_class, level=GROUPING_COL)
    count_per_year = subset.count(axis=0)
    mean_per_year = subset.mean(axis=0)
    sem_per_year = subset.sem(axis=0)
    mean_citations_per_year = subset.mean(axis=0)
    cit_dynamics_sharing_class_fig.add_trace(go.Scatter(
        name=share_class, legendgroup=share_class,
        x=mean_per_year.index,
        y=mean_per_year.values,
        error_y=dict(type='data', array=sem_per_year.values, width=5, visible=True,),
        marker=dict(
            color=CLASS_COLORS[share_class],
            size=count_per_year.values
        ),
        mode='lines+markers',
    ))

cit_dynamics_sharing_class_fig.update_layout(
    width=800, height=400,
    title=dict(
        text="Citation Dynamics: Different Types of Data Sharing",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top"
    ),
    xaxis=dict(
        title=dict(text="Years Since Publication", font=AXIS_TITLE_FONT, standoff=10),
        tickfont=AXIS_TICK_FONT,
        zeroline=False,
    ),
    yaxis=dict(
        title=dict(text="Mean Cumulative Citation per Year", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
        zeroline=False,
    ),
    legend=dict(
        title=dict(text="Data Sharing", font=AXIS_TITLE_FONT),
        font=LEGEND_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top",
        bgcolor='rgba(255,255,255,0.5)',
        bordercolor='black', borderwidth=1,
    ),
    margin=dict(t=75, b=50, l=50, r=25, pad=0),
    template="plotly_white"
)
cit_dynamics_sharing_class_fig.show()

In [ ]:
# flip to True to export this figure into `output/`
if False:
    save_figure(cit_dynamics_sharing_class_fig, "citation_dynamics_sharing_class.png", width=800, height=400)